In [11]:
import rasterio
from rasterio.windows import Window
from rasterio.features import shapes
from rasterio.windows import transform as window_transform
from rasterio.transform import xy

import numpy as np
import geopandas as gpd
import xarray as xr
import gc

from shapely.geometry import shape

In [12]:
out_tif = "/home/cassandra/Snohomish/2026-05-14/benchmark/max_depth.tif"
out_poly =  "/home/cassandra/Snohomish/2026-05-14/benchmark/max_depth.geojson"
DEM_tif = "/home/cassandra/Data/Snoh_DEM_composite/Snohomish_MosaicDEM_modded.tif"

qtr_file = "/home/cassandra/Snohomish/2026-05-14/sfincs.nc"
sbg_file = "/home/cassandra/Snohomish/2026-05-14/sfincs_subgrid.nc"
out_file = "/home/cassandra/Snohomish/2026-05-14/benchmark/sfincs_map.nc"

In [3]:
sbg = xr.open_dataset(sbg_file)
qtr = xr.open_dataset(qtr_file)
out = xr.open_dataset(out_file)

In [4]:
# Load grid min and max vals
zmin = sbg.z_zmin.values
zsmax = np.nanmax(out.zs.values, axis=0)

# Nan out dry cells
zsmax[zsmax <= zmin] = np.nan

/tmp/ipykernel_144689/368999291.py:3: RuntimeWarning: All-NaN slice encountered
  zsmax = np.nanmax(out.zs.values, axis=0)


In [5]:
# Load quadtree grid data
bx = qtr.mesh2d_node_x.values
by = qtr.mesh2d_node_y.values
i = qtr.mesh2d_face_nodes.values.astype(int)
lev = qtr.level.values

# Get the grid spacing
qdx = qtr.dx / (2** (lev - 1))
qdy = qtr.dy / (2** (lev - 1))

# Get the corners
x0, y0 = [], []
for j in range(len(i)):
    x0.append(bx[i[j][0]])
    y0.append(by[i[j][0]])
x0 = np.array(x0)
y0 = np.array(y0)

# Get the centers
xc, yc = x0 + qdx/2, y0 + qdy/2

/tmp/ipykernel_144689/863344822.py:4: RuntimeWarning: invalid value encountered in cast
  i = qtr.mesh2d_face_nodes.values.astype(int)


In [6]:
bdx, bdy = 1, 1
window_size = 8*1024

with rasterio.open(DEM_tif) as src:
    profile = src.profile.copy()
    crs = src.crs
    
    # Create output file with same metadata
    with rasterio.open(out_tif, 'w', **profile) as dst:
        
        # Iterate over windows
        for wyi in range(0, src.height, int(window_size)):
            print('')
            print("Chunk", int(wyi/window_size)+1, 'of', int(src.height/window_size)+1)
            for wxi in range(0, src.width, int(window_size)):
                print('_',end='')
            print('')
            for wxi in range(0, src.width, int(window_size)):
                print('.',end='')
                
                # Define a window
                w = Window(wxi, wyi,
                           min(window_size, src.width - wxi),
                           min(window_size, src.height - wyi))
                # Read data for the window
                zb = src.read(window=w)
                
                # Read x, y
                # Rows and columns for this window
                rows = np.arange(w.height)
                cols = np.arange(w.width)
                # Compute x coordinates (center of each column)
                win_transform = src.window_transform(w)
                x, _ = xy(win_transform, 0 * np.ones_like(cols), cols, offset='center')
                # Compute y coordinates (center of each row)
                _, y = xy(win_transform, rows, 0 * np.ones_like(rows), offset='center')
                x = np.array(x)
                y = np.array(y)
                
                # For each quadtree grid cell, locate and assign the relevant bathy indices (point from bathy --> quadtree)
                qi = np.int64(list(range(len(qtr.mesh2d_nFaces))))
                xi0 = np.int64((x0 - x[0]) / bdx)
                yi0t = np.int64((y0 - y[-1]) / bdy)
                xif = np.int64((x0 + qdx - x[0]) / bdx)
                yift = np.int64((y0 + qdy - y[-1]) / bdy)
                # flip y axis... I think...
                yif = -1 * yi0t + len(y) 
                yi0 = -1 * yift + len(y)
                
                # Check what intersects the area of interest -- full quadtree cells only
                qok = np.logical_and(np.logical_and(xif >= 0, xi0 < len(x)),
                                     np.logical_and(yif >= 0, yi0 < len(y)))
                qok = np.logical_and(qok, zsmax)
                
                # Access model output
                flood_depth = np.zeros(zb.shape)*np.nan
                for i in qi[qok]:
                    for xi in range(xi0[i],xif[i]):
                        if xi >= len(x) or xi < 0:
                            continue
                        for yi in range(yi0[i], yif[i]):
                            if yi >= len(y) or yi < 0:
                                continue
                            flood_depth[0,yi,xi] = zsmax[i] - zb[0,yi,xi]
                flood_depth[flood_depth < 0.01] = np.nan
                
                # Write to new geotif
                dst.write(flood_depth, window=w)
                # collect to try to keep memory leaks under control
                gc.collect()


Chunk 1 of 9
________
........
Chunk 2 of 9
________
........
Chunk 3 of 9
________
........
Chunk 4 of 9
________
........
Chunk 5 of 9
________
........
Chunk 6 of 9
________
........
Chunk 7 of 9
________
........
Chunk 8 of 9
________
........
Chunk 9 of 9
________
........

## Convert to Polygon

In [9]:
def tiff_to_poly(tiff_file, poly_file, cutoff_depth = 0.01, chunk_size=10000):
    geometries = []
    with rasterio.open(tiff_file) as src:
        crs = src.crs
        for row_off in range(0, src.height, chunk_size):
            for col_off in range(0, src.width, chunk_size):
                window = Window(col_off=col_off, row_off=row_off,
                                width=min(chunk_size, src.width - col_off),
                                height=min(chunk_size, src.height - row_off))
                data = src.read(1, window=window)
                if np.all(np.isnan(data)):
                    continue
                mask = data >= cutoff_depth
                if not np.any(mask):
                    continue
                win_transform = window_transform(window, src.transform)
                for geom, value in shapes(mask.astype(np.uint8), mask=mask, transform=win_transform):
                    if value == 1:
                        geometries.append(shape(geom))
    merged = gdf.geometry.union_all()
    merged_gdf = gpd.GeoDataFrame(geometry=[merged], crs=gdf.crs)
    merged_gdf = merged_gdf.explode(index_parts=False).reset_index(drop=True)
    merged_gdf.to_file(poly_file, driver='GeoJSON')

In [ ]:
tiff_to_poly(out_tif, out_poly)